# 001 · The math Module

> **Standard Library › Numbers & Math** · Python 3.14

`math` is Python's counterpart to `System.Math`: square roots, rounding, logarithms, trigonometry and constants — plus integer helpers (`gcd`, `comb`, `isqrt`, `factorial`) that .NET leaves to you or to `BigInteger`.

**In this notebook you will learn:**

- how `math` maps onto `System.Math`, and where the two differ (errors vs `NaN`, `int` vs `double` results)
- `pow` vs `**`, `floor`/`ceil`/`trunc`, and comparing floats with `isclose`
- accurate summation with `fsum`, plus `prod`, `dist` and `hypot`
- integer helpers: `gcd`, `lcm`, `factorial`, `comb`, `perm`, `isqrt`
- constants (`pi`, `e`, `tau`, `inf`, `nan`), logarithms, and `cmath` for complex numbers

## .NET mental model

| .NET | Python |
|---|---|
| `Math.Sqrt`, `Math.Pow`, `Math.Log`, `Math.Sin`... | `math.sqrt`, `math.pow`, `math.log`, `math.sin`... |
| `Math.PI`, `Math.E`, `Math.Tau` | `math.pi`, `math.e`, `math.tau` |
| `double.PositiveInfinity`, `double.NaN` | `math.inf`, `math.nan` |
| `Math.Abs`, `Math.Max`, `Math.Min`, `Math.Round` | built-ins `abs`, `max`, `min`, `round` (not in `math`) |
| `BigInteger.GreatestCommonDivisor` | `math.gcd` (Python `int` is already arbitrary precision) |
| `System.Numerics.Complex` | built-in `complex` + `cmath` module |

Two big behavioural differences:

1. `System.Math` usually **returns `NaN`** for invalid input (`Math.Sqrt(-1)`). Python's `math` **raises `ValueError`**.
2. `Math.Floor`/`Ceiling`/`Truncate` return a `double`. Python's `math.floor`/`ceil`/`trunc` return an **`int`**.

Remember also that Python `int` never overflows *(see Syntax › Data Types › 001 · Numbers)*.

In [ ]:
import math

print(math.sqrt(16), math.sqrt(2))
try:
    math.sqrt(-1)                 # C#: Math.Sqrt(-1) == double.NaN
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `math.pow` vs `**`

```csharp
double big = Math.Pow(2, 100);              // always double
var exact = BigInteger.Pow(2, 100);         // exact
```

`math.pow` always converts to `float` (like `Math.Pow`). The `**` operator (and built-in `pow`) keeps **exact integers** when both operands are `int`.

In [ ]:
print(math.pow(2, 10), 2 ** 10)        # 1024.0 vs 1024
print(math.pow(3, 40))                 # float — precision lost
print(3 ** 40)                         # exact int
print(int(math.pow(3, 40)) == 3 ** 40) # False!

**Tip:** prefer `**` in everyday code. Built-in `pow(base, exp, mod)` also does fast modular exponentiation (like `BigInteger.ModPow`):

In [ ]:
print(pow(7, 128, 13))     # (7 ** 128) % 13, computed efficiently

## `floor`, `ceil`, `trunc` — they return `int`

```csharp
double f = Math.Floor(-2.5);     // -3.0 (a double)
int i = (int)Math.Floor(-2.5);   // cast needed
```

In [ ]:
x = -2.5
print(math.floor(x), math.ceil(x), math.trunc(x))   # -3 -2 -2
print(type(math.floor(x)).__name__)                 # int — no cast needed
print(int(x))                                        # int() truncates toward zero, like (int) in C#

⚠️ **Gotcha:** `//` (floor division) rounds toward **negative infinity**, matching `math.floor`, while C#'s integer `/` truncates toward zero *(see Syntax › Operators › 001 · Arithmetic and Assignment Operators)*.

In [ ]:
print(-7 // 2, math.floor(-7 / 2), math.trunc(-7 / 2))   # -4 -4 -3

**Tip:** for rounding to *n* digits use the built-in `round()`. Like `Math.Round`'s default (`MidpointRounding.ToEven`), it uses banker's rounding: `round(2.5) == 2`. For money, use `decimal` *(see Standard Library › Numbers & Math › 003 · Decimal Arithmetic)*.

In [ ]:
print(round(2.5), round(3.5), round(3.14159, 2))

## Comparing floats: `isclose`

`0.1 + 0.2 != 0.3` in every language that uses IEEE-754 doubles. C# has no built-in tolerance comparison; you write `Math.Abs(a - b) < eps`. Python ships `math.isclose`:

- `rel_tol` (default `1e-09`) — tolerance relative to the larger value
- `abs_tol` (default `0.0`) — needed when comparing to **zero**

In [ ]:
print(0.1 + 0.2 == 0.3)                       # False
print(math.isclose(0.1 + 0.2, 0.3))           # True

print(math.isclose(1e-12, 0.0))               # False — relative tolerance of 0 is 0
print(math.isclose(1e-12, 0.0, abs_tol=1e-9)) # True

## Accurate sums: `fsum` vs `sum`

Adding floats one at a time accumulates rounding error. `math.fsum` tracks the lost low-order bits and returns the **correctly rounded** result.

Since Python 3.12, the built-in `sum()` also uses a compensated algorithm for floats, so it's far better than a naive loop — but `fsum` is still exact in cases where `sum` isn't.

```csharp
double total = 0;
foreach (var x in values) total += x;   // naive: error accumulates
```

In [ ]:
values = [0.1] * 10

total = 0.0
for v in values:          # naive loop, like the C# foreach
    total += v
print(total)               # 0.9999999999999999
print(sum(values))         # 1.0 (compensated since 3.12)
print(math.fsum(values))   # 1.0 (exact)

With values of wildly different magnitudes, only `fsum` gets it right:

In [ ]:
tricky = [1e100, 2.5, 1e16, -1e100, -1e16]   # the big values cancel out exactly
print(sum(tricky))         # 2.0 — the 2.5 was partly lost
print(math.fsum(tricky))   # 2.5 — correctly rounded exact result

## `prod`, `dist`, `hypot`

In [ ]:
print(math.prod([2, 3, 4]))                 # 24 — LINQ Aggregate((a, b) => a * b)
print(math.prod([1.5, 2], start=10))        # 30.0

print(math.hypot(3, 4))                     # 5.0 — length of a vector (any number of dims)
print(math.hypot(1, 2, 2))                  # 3.0
print(math.dist((1, 1), (4, 5)))            # 5.0 — Euclidean distance between two points

## Integer helpers

None of these exist on `System.Math`. All work on arbitrarily large integers.

| Function | Meaning |
|---|---|
| `gcd(*ints)` / `lcm(*ints)` | greatest common divisor / least common multiple (any number of args) |
| `factorial(n)` | n! |
| `comb(n, k)` | combinations — "n choose k" |
| `perm(n, k)` | ordered arrangements of k out of n |
| `isqrt(n)` | integer square root, `floor(sqrt(n))`, **exact** for huge `n` |

In [ ]:
print(math.gcd(12, 18, 24), math.lcm(4, 6, 10))     # 6 60
print(math.factorial(20))                            # fits in a long...
print(math.factorial(25))                            # ...this one wouldn't
print(math.comb(52, 5), "poker hands")
print(math.perm(10, 3), "podium finishes among 10 runners")

`isqrt` avoids the float round-trip, which loses precision for big numbers:

In [ ]:
n = 10**40 + 1
print(math.isqrt(n))          # exact integer
print(math.sqrt(n))           # float approximation

def is_perfect_square(k: int) -> bool:
    r = math.isqrt(k)
    return r * r == k

print(is_perfect_square(144), is_perfect_square(145))

## Constants, infinity and NaN

In [ ]:
print(math.pi, math.e, math.tau)       # tau == 2 * pi
print(math.inf > 10**300, -math.inf)   # inf compares with any number

nan = math.nan
print(nan == nan)                      # False — NaN is never equal to anything
print(math.isnan(nan), math.isinf(math.inf), math.isfinite(1.0))

⚠️ **Gotcha:** never test for NaN with `==` (same as C#, where `double.NaN == double.NaN` is `false`). Use `math.isnan(x)` (C#: `double.IsNaN(x)`).

## Logarithms and exponentials

| C# | Python |
|---|---|
| `Math.Log(x)` | `math.log(x)` (natural log) |
| `Math.Log(x, b)` | `math.log(x, b)` |
| `Math.Log10(x)` / `Math.Log2(x)` | `math.log10(x)` / `math.log2(x)` |
| `Math.Exp(x)` | `math.exp(x)` |
| — | `math.log1p(x)` / `math.expm1(x)` — accurate for tiny `x` |

In [ ]:
print(math.log(math.e), math.log(8, 2), math.log2(8), math.log10(1000))
print(math.log(1000, 10))       # 2.9999999999999996 — computed as log(x)/log(b)
print(math.exp(1), math.log1p(1e-10))

⚠️ **Gotcha:** `math.log(x, base)` divides two natural logs and can be slightly off. Use `log2`/`log10` for those bases — they're exact for exact powers.

## Trigonometry

Angles are in **radians**, as in .NET. Use `math.degrees`/`math.radians` to convert (.NET 7+ has `double.DegreesToRadians`).

In [ ]:
angle = math.radians(30)
print(round(math.sin(angle), 10), math.degrees(math.pi / 4))
print(math.atan2(1, 1))       # quadrant-aware arctangent, like Math.Atan2(y, x)

## `cmath` — complex numbers

Python has a built-in `complex` type (literal suffix `j`). `math` functions refuse complex results; `cmath` provides versions that return them — like `System.Numerics.Complex.Sqrt`.

In [ ]:
import cmath

print(cmath.sqrt(-1))          # 1j  (math.sqrt(-1) raises)
z = 3 + 4j
print(abs(z), z.real, z.imag)  # 5.0 3.0 4.0
print(cmath.polar(z))          # (magnitude, phase)

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Square root | `Math.Sqrt(x)` → `NaN` if negative | `math.sqrt(x)` → `ValueError` if negative |
| Power | `Math.Pow(a, b)` (double) | `a ** b` (exact for ints) / `math.pow` (float) |
| Floor / ceiling | `Math.Floor` → `double` | `math.floor` → `int` |
| Float equality | `Math.Abs(a - b) < eps` | `math.isclose(a, b, abs_tol=...)` |
| Accurate sum | — | `math.fsum(xs)` |
| Product | `Aggregate((a, b) => a * b)` | `math.prod(xs)` |
| Distance | hand-written | `math.dist(p, q)` / `math.hypot(*v)` |
| GCD / LCM | `BigInteger.GreatestCommonDivisor` / — | `math.gcd` / `math.lcm` |
| n choose k | — | `math.comb(n, k)` |
| Integer sqrt | — | `math.isqrt(n)` |
| Constants | `Math.PI`, `Math.E`, `Math.Tau` | `math.pi`, `math.e`, `math.tau` |
| NaN check | `double.IsNaN(x)` | `math.isnan(x)` |
| Complex sqrt | `Complex.Sqrt` | `cmath.sqrt` |

## Exercises

1. Compute the distance between the points `(2, 3)` and `(7, 15)` in two different ways.
2. Write `round_up_to(x, step)` that rounds a positive number **up** to the next multiple of `step` (e.g. `round_up_to(23, 5) == 25`), returning an `int`.
3. How many ways are there to choose 6 numbers from 49? What's the probability of matching all 6?
4. Compare `sum` and `math.fsum` on `[1e20, 1.0, -1e20]`. Do they differ? What does `fsum` guarantee that `sum` doesn't?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import math

# 1
print(math.dist((2, 3), (7, 15)))            # 13.0
print(math.hypot(7 - 2, 15 - 3))             # 13.0

# 2 — ceil returns an int already
def round_up_to(x, step):
    return math.ceil(x / step) * step

print(round_up_to(23, 5), round_up_to(25, 5))   # 25 25

# 3
ways = math.comb(49, 6)
print(ways, f"{1 / ways:.10f}")               # 13983816 0.0000000715

# 4 — since 3.12 sum() uses compensated summation, so it handles this one;
#     fsum is guaranteed to return the correctly rounded exact sum for ANY input
xs = [1e20, 1.0, -1e20]
print(sum(xs), math.fsum(xs))                 # 1.0 1.0
```

</details>